# Standardized coefficients, every model on one axes

One dodged forest plot per question, overlaying all eight runs on shared
NTP | FA axes. Reads the `full_standardized_coefficients.csv` files already in
`outputs/`, refits nothing, skips arms that were never run.

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = untuned `base`, solid = `german` LoRA |
| x | coefficient per 1 SD of predictor, with 95% CI |

Writes `figures/fig_model_standardized_coefficients_<question>.{png,pdf}` and
`outputs/culture/model_standardized_coefficients.csv`.

`.venv` has no jupyter and `uv run` must not be used here — it destroys the
hand-built sm_120 `llama-cpp-python` wheel. Run the code cells through
`.venv/bin/python`. Committed without cell outputs.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.figure import Figure

from culture.palette import (
    MIXTRAL_ARCHIVED,
    MIXTRAL_FRESH,
    REFERENCE_MARKERS,
    REFERENCE_TONES,
    arm_order,
    model_marker,
    model_tone,
)
from culture.registry import ARMS, CULTURE_MODELS, is_base
from culture.tables import read_csv
from machine_bias_reproduction.config import FIGURES_ROOT, OUTPUTS_ROOT, paths_for
from machine_bias_reproduction.figures import (
    CENTER_PREDICTOR,
    label_fit,
    predictor_rows,
    save_plate,
)
from machine_bias_reproduction.io_utils import write_csv
from machine_bias_reproduction.plates import GRID, MUTED_INK, bar_layout
from machine_bias_reproduction.questions import resolve_questions

COEFFICIENTS = "full_standardized_coefficients.csv"
FIT = "regression_fit.csv"
MODES = ("ntp", "fa")
XLABEL = "Coefficient per 1 SD of predictor — outcome log1p(nEMD)"
MODEL_INDEX = {key: index for index, key in enumerate(CULTURE_MODELS)}

## Finding the runs

`read_csv` returns `None` for a missing path — that is the whole
skip-what-was-never-run mechanism. The adjusted R² comes from the nested `full`
fit, the model the standardized refit rescales.

In [ ]:
def sources():
    found = [
        (f"{model.label} ({arm})", f"culture/{key}/{arm}", key, model.label, arm)
        for key, model in CULTURE_MODELS.items()
        for arm in arm_order(ARMS)
    ]
    found.append((MIXTRAL_ARCHIVED, "archived", None, MIXTRAL_ARCHIVED, None))
    found.append((MIXTRAL_FRESH, "fresh", None, MIXTRAL_FRESH, None))
    return found


def load_coefficients(question):
    frames = []
    for label, source, key, model_label, arm in sources():
        outputs = paths_for(source, question.var).outputs
        coefficients = read_csv(outputs / COEFFICIENTS)
        if coefficients is None:
            continue

        fit = read_csv(outputs / FIT)
        r_squared = {}
        if fit is not None:
            full = fit[fit["model"] == "full"]
            r_squared = dict(zip(full["mode"], full["adjusted_r_squared"], strict=True))

        frame = coefficients.drop(columns=["model"])
        frame.insert(0, "question", question.var)
        frame.insert(1, "question_label", question.label)
        frame.insert(2, "model_key", key)
        frame.insert(3, "model_label", model_label)
        frame.insert(4, "arm", arm)
        frame.insert(5, "series", label)
        frame["adjusted_r_squared"] = frame["mode"].map(r_squared)
        frames.append(frame)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## Encoding

The colour and marker each model already wears in `fig_model_culture_comparison`.
Dark2's orange and brown are too close to separate Gemma-4-E4B from Qwen3-VL-8B
by colour alone, so the model is doubly encoded and fill is left to carry the
arm. Mixtral fresh's near-white grey needs a darker rim to read at all.

In [ ]:
def series_style(label, model_key, arm):
    if label in REFERENCE_TONES:
        ink = REFERENCE_TONES[label].ink
        return {
            "marker": REFERENCE_MARKERS[label],
            "color": ink,
            "markerfacecolor": ink,
            "markeredgecolor": ink if label == MIXTRAL_ARCHIVED else MUTED_INK,
        }
    index = MODEL_INDEX[model_key]
    ink = model_tone(model_key, index).ink
    return {
        "marker": model_marker(model_key, index),
        "color": ink,
        "markerfacecolor": "none" if is_base(arm) else ink,
        "markeredgecolor": ink,
    }

## The plate

`predictor_rows` gives the grouped y-order, `bar_layout` the dodge.
`nEMD_center` gets its own strip because it dwarfs the demographic dummies.
Constrained layout, not tight: shared axes across gridspec rows of very unequal
height are what `tight_layout` warns it cannot solve.

In [ ]:
def draw_chrome(axis, rows):
    axis.axvline(0, color=MUTED_INK, linewidth=0.8)
    axis.set_yticks(range(len(rows)), [label for _, label in rows])
    axis.set_ylim(len(rows) - 0.5, -0.5)
    axis.grid(axis="x", color=GRID, linewidth=0.6)


def draw_series(axis, subset, rows, offset, legend):
    positions = [index for index, (predictor, _) in enumerate(rows) if predictor is not None]
    indexed = subset.set_index("predictor").reindex([rows[index][0] for index in positions])

    estimate = indexed["estimate"].to_numpy(dtype=np.float64)
    lower = estimate - indexed["ci_low"].to_numpy(dtype=np.float64)
    upper = indexed["ci_high"].to_numpy(dtype=np.float64) - estimate
    drawn = ~np.isnan(estimate)
    if not drawn.any():
        return

    label = subset["series"].iat[0]
    style = series_style(label, subset["model_key"].iat[0], subset["arm"].iat[0])
    axis.errorbar(
        estimate[drawn],
        np.asarray(positions, dtype=np.float64)[drawn] + offset,
        xerr=np.vstack([lower[drawn], upper[drawn]]),
        linestyle="none",
        markersize=4.0,
        markeredgewidth=1.0,
        ecolor=style["color"],
        capsize=1.5,
        elinewidth=0.8,
        label=label if legend else None,
        **style,
    )


def coefficient_plate(frame, question, destination):
    frame = frame[frame["predictor"] != "const"]
    labels = list(dict.fromkeys(frame["series"]))
    rows = predictor_rows(set(frame["predictor"]))
    has_center = CENTER_PREDICTOR in set(frame["predictor"])
    _, offsets = bar_layout(len(labels))

    pitch = max(0.34, 0.30 + 0.045 * len(labels))
    height = max(9.0, pitch * len(rows) + (1.6 if has_center else 0.0))
    figure = Figure(figsize=(15, height), layout="constrained")
    figure.get_layout_engine().set(hspace=0.015, wspace=0.015)
    grid = figure.add_gridspec(2, 2, height_ratios=[1, len(rows)]) if has_center \
        else figure.add_gridspec(1, 2)

    center_axes = []
    social_axes = []
    for column, mode in enumerate(MODES):
        panel = frame[frame["mode"] == mode]
        if has_center:
            axis = figure.add_subplot(
                grid[0, column],
                sharex=center_axes[0] if center_axes else None,
                sharey=center_axes[0] if center_axes else None,
            )
            center_rows = [(CENTER_PREDICTOR, r"$\bf{nEMD\ center}$")]
            draw_chrome(axis, center_rows)
            for label, offset in zip(labels, offsets, strict=True):
                draw_series(axis, panel[panel["series"] == label], center_rows, offset, False)
            label_fit(axis, mode.upper())
            axis.set_xlabel(XLABEL)
            center_axes.append(axis)

        axis = figure.add_subplot(
            grid[1 if has_center else 0, column],
            sharex=social_axes[0] if social_axes else None,
            sharey=social_axes[0] if social_axes else None,
        )
        draw_chrome(axis, rows)
        for label, offset in zip(labels, offsets, strict=True):
            draw_series(axis, panel[panel["series"] == label], rows, offset, column == 0)
        axis.set_xlabel(XLABEL)
        if not has_center:
            label_fit(axis, mode.upper())
        social_axes.append(axis)

    for axis in center_axes[1:] + social_axes[1:]:
        axis.tick_params(labelleft=False)

    handles, names = social_axes[0].get_legend_handles_labels()
    figure.legend(handles, names, ncol=min(4, len(names)), loc="outside lower center")
    figure.suptitle(f"{question.label} — standardized coefficients by model", fontsize=12)
    stem = destination / f"fig_model_standardized_coefficients_{question.var}"
    return save_plate(figure, stem)


def build():
    frames = []
    for question in resolve_questions(None):
        frame = load_coefficients(question)
        if frame.empty:
            continue
        frames.append(frame)
        coefficient_plate(frame, question, FIGURES_ROOT)

    table = pd.concat(frames, ignore_index=True)
    write_csv(table, OUTPUTS_ROOT / "culture" / "model_standardized_coefficients.csv")
    return table

## Build

In [ ]:
table = build()
print(len(table), "rows")

## Reading the plates

- Whiskers straddling 0 for every model: a predictor nobody leans on.
- Hollow and solid separating in one colour: the LoRA moving that model.
- Adjusted R² is in the CSV, not on the plate — eight series, no room.
- Absent arms contribute no marker, not a marker at zero.